# Spark File Formats — S3 Edition

This notebook explores how Spark reads CSV, JSON, and Parquet files from S3 and benchmarks their performance.

**Dataset schema**: `id`, `first_name`, `last_name`, `age`, `city`

**S3 layout**:
```
s3://emr-teaching-mentorship-2026/
  shared/datasets/        ← read-only datasets uploaded by instructor
  <your-name>/            ← your personal prefix, read/write
```

## 0. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder.appName("FileFormatsS3").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
# Update STUDENT_NAME to your own prefix (e.g. "anastasia" or "artur")
STUDENT_NAME = "anastasia"
BUCKET       = "emr-teaching-mentorship-2026"

SHARED = f"s3://{BUCKET}/shared/datasets"
MY_PREFIX = f"s3://{BUCKET}/{STUDENT_NAME}"

CSV_PATH     = f"{SHARED}/data.csv"
JSON_PATH    = f"{SHARED}/data.json"
PARQUET_PATH = f"{SHARED}/data.parquet"

print(f"Reading from : {SHARED}")
print(f"Writing to   : {MY_PREFIX}")

## 1. Read all three formats

In [ ]:
df_csv     = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
df_json    = spark.read.json(JSON_PATH)
df_parquet = spark.read.parquet(PARQUET_PATH)

print("CSV schema:")
df_csv.printSchema()
df_csv.show(5)

## 2. Full-scan benchmark (read + count)

In [ ]:
def bench(label, df):
    t0 = time.time()
    n  = df.count()
    elapsed = time.time() - t0
    print(f"{label:10s}  rows={n:,}  time={elapsed:.2f}s")
    return elapsed

print("=== Full scan ===")
t_csv     = bench("CSV",     df_csv)
t_json    = bench("JSON",    df_json)
t_parquet = bench("Parquet", df_parquet)

## 3. Filter + Select benchmark

Parquet stores column statistics (min/max per row-group) so Spark can skip entire chunks without reading them.

In [ ]:
TARGET_CITY = "Springfield"

def bench_filter(label, df):
    filtered = df.select("city", "first_name").filter(df.city == TARGET_CITY)
    t0 = time.time()
    n  = filtered.count()
    elapsed = time.time() - t0
    print(f"{label:10s}  matches={n:,}  time={elapsed:.2f}s")
    return elapsed

print(f"=== Filter: city == '{TARGET_CITY}' ===")
bench_filter("CSV",     df_csv)
bench_filter("JSON",    df_json)
bench_filter("Parquet", df_parquet)

## 4. Explain plans — what is Spark actually doing?

`.explain(mode="formatted")` shows the physical plan Spark will execute.

In [ ]:
print("=== CSV plan ===")
df_csv.select("city", "first_name").filter(df_csv.city == TARGET_CITY).explain(mode="formatted")

In [ ]:
print("=== Parquet plan ===")
df_parquet.select("city", "first_name").filter(df_parquet.city == TARGET_CITY).explain(mode="formatted")

**Things to spot in the Parquet plan**:
- `PushedFilters` — predicate pushed into the reader, Spark skips row-groups before loading them
- `ReadSchema` — only selected columns are read (columnar storage)

## 5. Aggregation — age statistics per city

In [ ]:
stats = (
    df_parquet
    .groupBy("city")
    .agg(
        F.count("*").alias("total"),
        F.avg("age").alias("avg_age"),
        F.min("age").alias("min_age"),
        F.max("age").alias("max_age")
    )
    .orderBy(F.desc("total"))
)

stats.show(10)

## 6. Write results back to your S3 prefix

Write the aggregated stats as Parquet to your personal S3 prefix.

In [ ]:
output_path = f"{MY_PREFIX}/output/city_age_stats"

stats.write.mode("overwrite").parquet(output_path)
print(f"Written to {output_path}")

In [ ]:
# Verify round-trip
readback = spark.read.parquet(output_path)
readback.show(5)
print("Row count:", readback.count())

## 7. Sorted Parquet — does sort order affect read performance?

When data is sorted by the filter column and written with Parquet's row-group statistics, Spark can skip entire row-groups.

In [ ]:
SORTED_PATH = f"{SHARED}/data_sorted_by_city.parquet"

try:
    df_sorted = spark.read.parquet(SORTED_PATH)

    print("=== Sorted Parquet filter benchmark ===")
    t0 = time.time()
    n  = df_sorted.filter(df_sorted.city == TARGET_CITY).count()
    print(f"sorted parquet  matches={n:,}  time={time.time()-t0:.2f}s")
    print(f"unsorted parquet was {t_parquet:.2f}s")
except Exception as e:
    print(f"Sorted file not found — instructor may not have uploaded it yet: {e}")

---

## Homework

Answer each question with working code in the cell below it.

---

### Q1 — Largest cities
Using the **Parquet** dataset, find the **top 5 cities** by number of people in the dataset.  
Show city name and count, sorted descending.

In [ ]:
# Q1 — your answer here


### Q2 — Young adults
How many people in the dataset are aged **18–25** (inclusive)?  
Use CSV, JSON, and Parquet. Record the time for each.

In [ ]:
# Q2 — your answer here


### Q3 — Explain the difference
Run `.explain(mode="formatted")` on the Q2 filter for both CSV and Parquet.  
In the Markdown cell below, write 2–3 sentences explaining what `PushedFilters` means and why Parquet benefits from it while CSV does not.

In [ ]:
# Q3 — explain() calls here


**Q3 written answer**:  
_your explanation here_

### Q4 — Write your own Parquet
Filter the dataset to only people whose **last name starts with the letter 'S'**.  
Write the result as Parquet to `s3://<bucket>/<your-name>/homework/last_name_S/`.  
Read it back and confirm the row count.

In [ ]:
# Q4 — your answer here


### Q5 — Bonus: Partitioned write
Write the full Parquet dataset **partitioned by city** to `s3://<bucket>/<your-name>/homework/partitioned_by_city/`.  
Then read it back with a filter `city == 'Springfield'` and call `.explain()`.  
What is different about the plan compared to the non-partitioned read?

In [ ]:
# Q5 — your answer here


**Q5 written answer**:  
_your explanation here_

In [ ]:
spark.stop()